### **IMPORT NECESSARY LIBRARIES**

In [87]:
!pip install emoji --quiet

In [88]:
import pandas as pd
import numpy as np
import re
import string
import emoji
import nltk
from nltk.corpus import stopwords
from google.colab import files

In [89]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### **LOAD DATASET**

In [90]:
df = pd.read_csv('/content/expanded_dataset_final.csv')
print("Shape before cleaning:", df.shape)
print("\nColumns:", df.columns.tolist())
display(df.head())

Shape before cleaning: (201551, 3)

Columns: ['ID', 'Rating', 'Review']


,ID,Rating,Review
0,0.0,5,Best under 60k Great performanceI got it for a...
1,1.0,5,Good perfomence...
2,2.0,5,Great performance but usually it has also that...
3,3.0,5,My wife is so happy and best product 👌🏻😘
4,4.0,5,"Light weight laptop with new amazing features,..."


### **PREPROCESSING TECHNIQUES**

**1. Handle Missing Data**

In [91]:
# Handle Missing Data
df.dropna(subset=['Review', 'Rating'], inplace=True)
df['Review'] = df['Review'].fillna('')
df.shape

(201538, 3)

In [92]:
df = df[df['Rating'].apply(lambda x: str(x).isdigit() and 1 <= int(x) <= 5)]
df['Rating'] = df['Rating'].astype(int)
print(f"Shape after filtering invalid ratings: {df.shape}")

Shape after filtering invalid ratings: (201534, 3)


In [93]:
# Keep only rows with integer Ratings
#df = df[df['Rating'].apply(lambda x: isinstance(x, (int, np.integer)))]
#print(f"Shape after keeping only integer Ratings: {df.shape}")

In [94]:
# Drop Empty or Too Short/Long Reviews (3–100 words)
df['word_count'] = df['Review'].apply(lambda x: len(str(x).split()))
df = df[(df['word_count'] >= 3) & (df['word_count'] <= 100)].copy()

In [95]:
# Basic Feature Engineering
df['review_length'] = df['Review'].apply(len)

In [96]:
# Remove Duplicates & Conflicts
df.drop_duplicates(subset=['Review', 'Rating'], inplace=True)
conflict_mask = df.duplicated(subset=['Review'], keep=False)
conflicts = df[conflict_mask]
df = df[~df['Review'].isin(conflicts['Review'])]

In [97]:
# Text Cleaning
stop_words = set(stopwords.words('english'))
removed_stopwords = []  # to collect all stopwords removed

def clean_text(text):
    text = text.lower()                                            # lowercase
    text = emoji.replace_emoji(text, replace='')                   # remove emojis
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)            # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)                          # remove symbols/numbers
    text = re.sub(r'\s+', ' ', text).strip()                       # remove extra spaces

    words = text.split()
    filtered_words = []
    for w in words:
        if w not in stop_words:
            filtered_words.append(w)
        else:
            removed_stopwords.append(w)

    return ' '.join(filtered_words)

# Apply cleaning
df['Cleaned_Review'] = df['Review'].astype(str).apply(clean_text)

In [98]:
# Stopword Removal Summary
from collections import Counter
stopword_counts = Counter(removed_stopwords)
unique_removed = list(stopword_counts.keys())

print("\n-STOPWORD REMOVAL SUMMARY-")
print(f"Total stopwords removed: {len(removed_stopwords):,}")
print(f"Unique stopwords removed: {len(unique_removed)}")
print("\nList of stopwords removed:")
print(unique_removed)


-STOPWORD REMOVAL SUMMARY-
Total stopwords removed: 586,282
Unique stopwords removed: 147

List of stopwords removed:
['under', 'it', 'for', 'is', 'but', 'to', 'its', 'very', 'this', 'with', 'so', 'if', 'any', 'has', 'that', 's', 'of', 'can', 'only', 'i', 'you', 'are', 'or', 'my', 'and', 'am', 'over', 'all', 'a', 'not', 'in', 'was', 'few', 'other', 'same', 'about', 'these', 'doesn', 't', 'm', 'll', 'again', 'after', 'as', 'now', 'the', 'because', 'once', 'doing', 'more', 'than', 'just', 'don', 'at', 'your', 'then', 'will', 'me', 'those', 'who', 'have', 'no', 'up', 'some', 'why', 'didn', 'on', 'from', 'an', 'they', 'here', 're', 'yourself', 'when', 'be', 'while', 'there', 'which', 'do', 'isn', 'such', 'own', 've', 'above', 'both', 'we', 'should', 'what', 'between', 'too', 'd', 'having', 'off', 'by', 'how', 'haven', 'our', 'been', 'won', 'her', 'before', 'most', 'down', 'during', 'were', 'o', 'nor', 'below', 'into', 'does', 'out', 'he', 'had', 'being', 'couldn', 'y', 'where', 'she', 'th

In [99]:
# Save all removed stopwords to TXT
with open("all_removed_stopwords.txt", "w") as f:
    for word in removed_stopwords:
        f.write(f"{word}\n")

files.download("all_removed_stopwords.txt")
print("\nAll removed stopwords saved to 'all_removed_stopwords.txt'")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All removed stopwords saved to 'all_removed_stopwords.txt'


In [100]:
# Final Cleanup
df.drop_duplicates(subset=['Cleaned_Review'], inplace=True)
df.reset_index(drop=True, inplace=True)

In [101]:
# Save & Download Cleaned Dataset
output_file = "cleaned_dataset_final.csv"
df.to_csv(output_file, index=False)
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [102]:
print("\nCleaning Completed Successfully!")
print("Final Shape:", df.shape)
print("Saved file:", output_file)


Cleaning Completed Successfully!
Final Shape: (82073, 6)
Saved file: cleaned_dataset_final.csv
